# CAS Exam 5: Reserving in Different Situations

**Source:** Friedland, J. *Estimating Unpaid Claims Using Basic Techniques*, CAS, 2010 — Chapter 20

**Exam task:** B — Identify and adjust for changes in the environment; reserve in special situations

**Learning goals:**
1. Explain why standard development methods need modification in special situations
2. Apply WC-specific adjustments: benefit changes, COLA, medical/indemnity separation
3. Describe how claims-made policies change the development pattern and IBNR structure
4. Analyze gross vs. net triangles for high-deductible programs
5. Understand the leverage effect in excess of loss reserving
6. Identify the key considerations for mass tort and asbestos/environmental reserving

> **See also:** `exam5_claimsmade_ratemaking.ipynb` for claims-made pricing; `exam5_recoveries_reinsurance.ipynb` for gross/ceded/net mechanics.

## Formula Sheet Quick Reference

| Concept | Key Formula / Approach |
|---|---|
| **WC benefit change adjustment** | Adjusted losses = Actual losses × (New benefit level / Old benefit level) for each period |
| **WC COLA factor** | Restate losses to current benefit level using COLA escalation factors |
| **Claims-made % IBNR** | Near zero pure IBNR; only IBNER (case reserve development on known claims) |
| **Gross vs. net IBNR** | Net IBNR = Gross IBNR − Ceded IBNR (project gross and ceded triangles separately) |
| **Excess layer loss** | Excess = max(loss − retention, 0) — leveraged volatility vs. ground-up |
| **Indicated IBNR for excess** | Higher % of expected losses than ground-up (large losses develop more slowly) |

## Section 1: Why Standard Methods Need Modification

The standard chain ladder, BF, and other reserving methods assume that the historical development pattern is **representative** of future development. This assumption breaks down when:

1. **External forces change** the cost per claim (e.g., WC benefit legislation)
2. **Coverage structure** limits what the insurer is actually paying (e.g., deductibles, reinsurance)
3. **Reporting trigger** differs from the standard accident-year basis (e.g., claims-made)
4. **Loss distribution** is truncated or leveraged (e.g., excess layers, self-insured retentions)
5. **Liabilities are contested or evolving** (mass torts, asbestos, environmental)

In each case, directly applying development factors to unadjusted historical data would produce **biased** reserve estimates. The actuary must identify the distortion, adjust the data, and/or use a method appropriate for the situation.

| Situation | Primary Distortion | Adjustment Approach |
|---|---|---|
| Workers' Compensation | Benefit changes, COLA | Restate triangle to current benefit level |
| Claims-Made | No pure IBNR tail | Use report-year triangle; focus on IBNER |
| High Deductible | Net ≠ Gross development | Project gross and ceded separately |
| Excess of Loss | Leveraged volatility | Use loss elimination ratios or ILF adjustments |
| Mass Tort / Long-Tail Environmental | Non-stationary patterns | Benchmarks, exposure models, scenario analysis |

## Section 2: Workers' Compensation

### Why WC Is Different

Workers' compensation has several characteristics that require special handling:

1. **Statutory benefits** — indemnity benefits are set by law and change when legislation passes. Historical triangles reflect old benefit levels; future development will reflect new (usually higher) benefit levels.
2. **COLA provisions** — many states require cost-of-living adjustments for long-term disability claimants, creating wage-inflation-driven reserve development.
3. **Medical vs. indemnity** — WC losses split into medical (health care costs) and indemnity (wage replacement). These have different development patterns and trend rates.
4. **Long development tail** — serious injury claims (permanent total disability, serious medical) develop for decades.
5. **Reopened claims** — WC claims can be legally reopened years after closure in some states.

### Benefit Change Adjustments

When WC benefits change (e.g., maximum weekly benefit increases), historical losses must be **restated** to the new benefit level before development factors are computed — otherwise, the loss triangle shows an artificial step change at the calendar year when the law changed.

**Approach:**
1. Identify the **effective date** of the benefit change
2. For each accident year-development age cell, determine which benefit level applies to the payments in that diagonal
3. Apply a **benefit change factor (BCF)** to restate old-law payments to new-law equivalents:

$$\text{Adjusted Loss}_{AY, age} = \text{Actual Loss}_{AY, age} \times \text{BCF}$$

The BCF is typically determined from workers' compensation rate filings or actuarial analysis of the statutory changes.

### COLA Adjustments

For states with COLA provisions:
- Permanent total disability (PTD) claims receive annual inflation adjustments to their weekly benefit
- These adjustments create reserve increases on older, open claims even with no underlying change in the claimant's condition
- To remove this distortion, restated the triangle by applying **COLA factors** to normalize all payments to the current benefit level

### Medical vs. Indemnity Separation

Best practice for WC: **develop medical and indemnity components separately**, then add.

| Component | Primary Trend Driver | Development Pattern |
|---|---|---|
| **Indemnity** | Wage growth, benefit changes, COLA | Moderate tail; dominated by permanent disability claims |
| **Medical** | Medical inflation, treatment patterns, managed care | Longer tail for serious injuries; highly sensitive to managed care penetration |

> **Exam tip** — The exam may ask you to identify which component (medical or indemnity) is more affected by a given change. Medical is more sensitive to healthcare cost trends; indemnity is more sensitive to benefit changes and COLA.

In [ ]:
import pandas as pd
import numpy as np

# Example: WC benefit change adjustment
# Suppose a benefit change effective 1/1/2023 increased benefits by 15%.
# Accident years prior to 2023 had payments under the old benefit level.
# We restate historical triangles to the new benefit level.

benefit_change_factor = 1.15  # new law / old law

# Sample cumulative paid loss triangle (old-law basis for AY 2020-2022)
triangle_raw = pd.DataFrame(
    {
        12: [1_200, 1_350, 1_500, 1_620],
        24: [1_680, 1_890, 2_100, None],
        36: [1_932, 2_173, None, None],
        48: [2_051, None, None, None],
    },
    index=[2020, 2021, 2022, 2023]
)

print("Raw Triangle ($000s):")
display(triangle_raw)

# Benefit change effective 1/1/2023.
# Payments in calendar year 2023 and later are already on the new-law basis.
# Payments in CY 2022 and earlier are on the old-law basis.
# Calendar year = accident year + development age (months) / 12
# Apply BCF to cells where the diagonal (CY) < 2023 (old-law payments)

triangle_adj = triangle_raw.copy().astype(float)
for ay in triangle_adj.index:
    for age in triangle_adj.columns:
        if pd.notna(triangle_adj.loc[ay, age]):
            cy = ay + age / 12  # approximate calendar year
            if cy < 2023:
                triangle_adj.loc[ay, age] *= benefit_change_factor

print("\nAdjusted Triangle (restated to new benefit level, $000s):")
display(triangle_adj.round(0))

# Compute age-to-age factors on adjusted triangle
ages = [12, 24, 36]
ldfs = {}
for i in range(len(ages)):
    a_from = ages[i]
    a_to = ages[i+1] if i+1 < len(ages) else 48
    valid = triangle_adj[[a_from, a_to]].dropna()
    if len(valid) > 0:
        ldfs[(a_from, a_to)] = valid[a_to].sum() / valid[a_from].sum()

print("\nAge-to-Age Factors (on adjusted triangle):")
for k, v in ldfs.items():
    print(f"  {k[0]}→{k[1]}: {v:.3f}")

## Section 3: Claims-Made Reserving

### How CM Changes the Development Triangle

Under a **claims-made policy**, the coverage trigger is when the claim is *reported* — not when the loss occurred. This fundamentally changes the IBNR structure:

**Key difference from occurrence:**
- Under occurrence, IBNR includes both unreported claims (pure IBNR) and reserve development on known claims (IBNER)
- Under claims-made, **pure IBNR is essentially zero** at any valuation date, because all claims covered by the policy *must* have been reported to the insurer during the policy period
- Only **IBNER** remains — the difference between case reserves and ultimate settlement values on already-reported claims

| Reserve Component | Occurrence | Claims-Made |
|---|---|---|
| **Pure IBNR** (unreported claims) | Significant, especially early in development | Near zero |
| **IBNER** (case reserve development) | Present | Present and dominates |
| **Development tail** | Long (claims emerge for years) | Short (limited to IBNER on known claims) |

### Using a Report-Year Triangle

For CM policies, organize the data as a **report-year** triangle:
- Rows = year in which the claim was *reported* (not accident year)
- Columns = development age in months from the report year
- Values = cumulative paid or incurred losses on claims reported in that year

The report-year development factors apply the chain ladder to IBNER only. They are typically lower than accident-year development factors for the same line because the pure-IBNR component is removed.

### First-Year vs. Mature Claims-Made

When a CM program is first established:
- **Report Year 1**: only claims from AY = current year that are reported in year 1 (fast reporters). This cohort has a limited claim count and may have atypical development.
- **Report Year 5+**: a mix of accident years — many prior years contributing reports. Development is more stable.

Actuaries must be careful applying all-year average development factors to immature (early-step) CM years.

> **Exam tip:** A question may ask why a CM triangle shows less development than an occurrence triangle for the same line. The answer: no pure IBNR on CM; pure IBNR represents a major portion of occurrence development at early ages.

## Section 4: High-Deductible and Large-Deductible Programs

### The Structure

Under a **high-deductible (HD) program**, the insured retains claims up to a per-occurrence deductible (e.g., \$250,000 or \$1 million). The insurer:
1. Pays claims on a **gross (first-dollar) basis** to claimants (for administrative convenience and regulatory compliance)
2. Then **recovers** the deductible portion from the insured
3. Holds collateral (letter of credit, trust fund) from the insured to secure potential recovery

**Accounting for the insurer:**
- **Gross reserve**: the insurer's total obligation to claimants (unlimited by deductible)
- **Ceded/deductible reserve**: amounts the insurer expects to recover from the insured
- **Net reserve** = Gross reserve − Ceded deductible reserve

### Why Net ≠ Gross Development

Claims often **develop differently at the ground-up (gross) level vs. the net level**:

| Factor | Effect on Development Difference |
|---|---|
| **Severity distribution** | Small claims (within deductible) close quickly; large claims take longer. Net = only large losses → longer-tailed net development |
| **Managed care incentives** | Insurer manages gross claims but may have less incentive to minimize deductible-layer costs → different closure patterns |
| **Reopened claims** | Reopened claims are more likely to exceed the deductible (large claims) → net IBNR higher % of gross |
| **Insured insolvency risk** | If the insured cannot fund the deductible, the ceded reserve is uncollectible → net reserve actually increases |

### Two Approaches

**Approach 1: Project gross and ceded separately**
- Build two triangles: gross and ceded (deductible recoveries)
- Apply development methods to each independently
- Net IBNR = Gross IBNR − Ceded IBNR
- Better accuracy when the gross and deductible layers have very different development patterns

**Approach 2: Project net directly**
- Build a net triangle (gross less deductible payments actually made)
- Apply development methods directly
- Simpler, but may mix closed small claims (fully within deductible) with open large claims → distorted development

> **Friedland's guidance:** Prefer Approach 1 for large deductibles and volatile loss distributions. Use Approach 2 only when the deductible layer is stable and well-understood.

### Collateral and Uncollectible Deductibles

When the insured fails or cannot fund the deductible:
- The insurer is still legally required to pay claimants
- The ceded reserve becomes uncollectible
- **Net reserve = Gross reserve** in this worst case

Actuaries must consider credit risk on the deductible receivable and may need to estimate a provision for **uncollectible deductibles** as a separate reserve component.

In [ ]:
import pandas as pd
import numpy as np

# High-deductible example: Project gross and ceded separately
# Per-occurrence deductible: $500,000

# Cumulative paid losses (gross, $000s)
gross_paid = pd.DataFrame(
    {12: [3_800, 4_100, 4_500, 4_900],
     24: [5_800, 6_200, 6_800, None],
     36: [7_000, 7_400, None, None],
     48: [7_420, None, None, None]},
    index=[2021, 2022, 2023, 2024]
)

# Cumulative deductible recoveries (ceded, $000s)
ceded_paid = pd.DataFrame(
    {12: [800, 850, 900, 950],
     24: [1_450, 1_550, 1_650, None],
     36: [1_800, 1_920, None, None],
     48: [1_960, None, None, None]},
    index=[2021, 2022, 2023, 2024]
)

def vol_wtd_ldf(triangle_df):
    ages = sorted([c for c in triangle_df.columns if isinstance(c, int)])
    ldfs = {}
    for i in range(len(ages)-1):
        a1, a2 = ages[i], ages[i+1]
        valid = triangle_df[[a1, a2]].dropna()
        if len(valid) > 0:
            ldfs[(a1, a2)] = valid[a2].sum() / valid[a1].sum()
    return ldfs

gross_ldfs = vol_wtd_ldf(gross_paid)
ceded_ldfs = vol_wtd_ldf(ceded_paid)

print("Gross development factors:")
for k,v in gross_ldfs.items(): print(f"  {k[0]}→{k[1]}: {v:.3f}")

print("\nCeded (deductible) development factors:")
for k,v in ceded_ldfs.items(): print(f"  {k[0]}→{k[1]}: {v:.3f}")

# The ceded LDFs are higher early on — deductible-layer claims (large losses) develop more slowly
# than ground-up — consistent with theory

# Project to ultimate (simplified: assume no tail beyond age 48)
# Gross: AY 2024 latest = 12 months; project using 12→24→36→48 LDFs
def project_ay(latest_val, latest_age, ldfs_dict):
    ages = sorted(ldfs_dict.keys(), key=lambda x: x[0])
    val = latest_val
    for (a1, a2) in ages:
        if a1 >= latest_age:
            val *= ldfs_dict[(a1, a2)]
    return val

results = []
for ay in [2021, 2022, 2023, 2024]:
    latest_age = max(a for a in [12,24,36,48] if pd.notna(gross_paid.loc[ay, a]))
    gross_latest = gross_paid.loc[ay, latest_age]
    ceded_latest = ceded_paid.loc[ay, latest_age]
    gross_ult = project_ay(gross_latest, latest_age, gross_ldfs)
    ceded_ult = project_ay(ceded_latest, latest_age, ceded_ldfs)
    results.append({'AY': ay, 'Gross IBNR': gross_ult - gross_latest,
                    'Ceded IBNR': ceded_ult - ceded_latest,
                    'Net IBNR': (gross_ult - gross_latest) - (ceded_ult - ceded_latest)})

res_df = pd.DataFrame(results)
res_df = pd.concat([res_df, res_df.sum().rename('Total').to_frame().T.assign(AY='Total')], ignore_index=True)
print("\nIBNR by Component ($000s):")
display(res_df.set_index('AY').round(0))

## Section 5: Excess of Loss Reserving

### The Leverage Effect

When reserving for **excess of loss** layers (e.g., claims above \$1 million, or aggregate stop-loss), the loss distribution is **truncated** below the attachment point. This creates a leverage effect:

- A 10% increase in the ground-up loss can produce a 30–50% increase in excess layer losses
- Excess layer losses are highly sensitive to the shape of the tail of the loss distribution
- Development factors for excess layers are typically **higher** than ground-up development factors because:
  1. Large claims (those that reach the excess layer) take longer to settle
  2. Reserve adequacy on large claims is less reliable — they are more likely to be deficient
  3. The percentage of IBNR is higher: a larger fraction of ultimate excess losses are unreported at any valuation date

### Two Methods for Excess Reserving

**Method 1: Develop excess layer directly**
- Build a triangle of excess losses only (losses above attachment point)
- Apply development factors derived from that excess triangle
- Problem: sparse data — few claims exceed the attachment point → volatile development factors

**Method 2: Ground-up with ILF adjustment**
- Project ground-up losses to ultimate using a full (ground-up) triangle
- Apply **increased limits factors (ILF)** to derive expected excess losses
- Less volatile but relies on the ILF assumptions being correct for this line and insured

**Method 3: Burn cost approach**
- Focus on claims that have already reached the excess layer ("burning" through the retention)
- Project these claims' ultimate costs using claim-specific projections
- Estimate IBNR as a loading on the developed known excess claims

### Why Excess Development Patterns Differ

| Age | Ground-Up LDF | Excess Layer LDF (example) | Why |
|---|---|---|---|
| 12→24 | 1.40 | 1.85 | Slow-reported large claims entering the excess layer |
| 24→36 | 1.18 | 1.40 | Large-claim severity development (reserve deficiency) |
| 36→48 | 1.06 | 1.15 | Ongoing litigation on complex/large claims |
| 48→ult | 1.02 | 1.05 | Final settlement of remaining large open claims |

> **Exam tip:** If asked whether IBNR as a % of ultimate is higher or lower for excess vs. ground-up, the answer is **higher for excess**. Both the pure IBNR % and the IBNER % are elevated in excess layers due to the leverage effect and slower development of large claims.

## Section 6: Self-Insured Retentions and Assumed Reinsurance

### Self-Insured Retentions (SIRs)

A **self-insured retention (SIR)** is similar to a large deductible but with one key difference:

| Feature | Large Deductible | SIR |
|---|---|---|
| Insurer pays first | Yes — insurer pays gross, recovers deductible | No — insured handles claims up to retention |
| Claims management | Insurer manages all claims | Insured manages SIR-layer claims |
| Insurer liability | Above deductible (net) | Above SIR only |
| Collateral | Typically required | Required by insurer |

For SIR programs:
- The insurer's triangle should reflect only losses **above** the SIR
- Development factors may differ from ground-up due to the same leverage effects as excess layers
- The insured's SIR layer should be reserved separately (typically by the insured's own actuary or TPA)

### Assumed Reinsurance

When an insurer **assumes** reinsurance from ceding companies:
- Reporting lags are typically longer (ceding companies report to reinsurers with a delay)
- IBNR on assumed business is higher as a percentage than for direct business
- Premium volume may not be known at valuation — additional IBNR for **premiums not yet reported**
- Catastrophe events create spikes in assumed reinsurance development that are absent in direct-only triangles

### Mass Tort / Asbestos / Environmental (A&E)

These special liabilities share characteristics that make standard development methods unreliable:

| Characteristic | Implication |
|---|---|
| **Latency** | Claims emerge decades after exposure — pure IBNR is enormous and uncertain |
| **Non-stationarity** | New claimants, new diseases, new legal theories continue to emerge |
| **Retroactive coverage disputes** | Coverage is litigated; expected recoveries from prior policies are uncertain |
| **No credible development pattern** | Cannot rely on triangle-based development; patterns are distorted by settlement waves |

**Friedland's guidance for A&E:** Standard development methods should not be used in isolation. Actuaries typically:
1. Use **exposure-based models** (# of sites, # of claimants exposed)
2. Reference **industry benchmarks** for A&E reserve adequacy
3. Perform **scenario analysis** around key assumptions (claim emergence rate, average settlement value)
4. Apply significant **actuarial judgment** and disclose uncertainty prominently

## Section 7: Practice Problems

### Problem 1 — Workers' Compensation Benefit Change

A state increases WC indemnity benefits by 12% effective January 1, 2024. An insurer has the following cumulative paid indemnity triangle (CY payments made through December 31, 2024, $000s):

| AY | 12 mo | 24 mo | 36 mo |
|---|---|---|---|
| 2022 | 900 | 1,260 | 1,450 |
| 2023 | 980 | 1,370 | — |
| 2024 | 1,050 | — | — |

All 12-month and 24-month payments for AY 2022 and 2023 were made at the old benefit level. The 36-month diagonal (December 31, 2024) for AY 2022 includes payments at the new benefit level.

**(a)** Restate the entire triangle to the new (post-2024) benefit level by applying the 12% increase to all old-law payments.

**(b)** Compute volume-weighted age-to-age factors from the adjusted triangle.

**(c)** Using a simple 1.050 tail factor, estimate the AY 2024 ultimate indemnity losses.

---

### Problem 2 — Claims-Made vs. Occurrence

**(a)** An E&O insurer converts a book from occurrence to claims-made coverage at the start of 2024. At year-end 2024, the claims-made triangle shows very little IBNR development compared to the prior occurrence triangle for the same line. Explain whether this is expected and why.

**(b)** A claims-made actuary is evaluating a 5-year-old book of claims-made E&O policies. At year-end 2024 (report year 2024), the case reserves appear adequate. What does the IBNR reserve represent for this book? Should it be larger or smaller than the IBNR reserve on a comparable occurrence book? Explain.

**(c)** For the same book, a large claim from accident year 2020 was reported in report year 2022 and is now settling for 3× the original case reserve. What reserve component does this update represent, and what method would best capture it?

---

### Problem 3 — High-Deductible Program

An insurer has a workers' compensation account with a \$500,000 per-occurrence deductible. The following data is available as of year-end 2024:

| | Gross IBNR | Ceded (Deductible) IBNR |
|---|---|---|
| Projected | \$4,200,000 | \$1,600,000 |

The insured is in financial distress and there is a 20% probability they will be unable to fund the deductible.

**(a)** What is the "base" net IBNR (assuming full collectibility)?

**(b)** What is the **expected** net IBNR after reflecting the 20% insolvency probability?

**(c)** How should the actuary communicate this uncertainty in the reserve opinion?

---

### Solutions

<details>
<summary>Click to reveal solutions</summary>

**Problem 1:**  
(a) Restate old-law payments: multiply all cells EXCEPT the AY 2022 age-36 cell by 1.12:

| AY | 12 mo | 24 mo | 36 mo |
|---|---|---|---|
| 2022 | 1,008 | 1,411 | 1,450 |
| 2023 | 1,098 | 1,534 | — |
| 2024 | 1,176 | — | — |

(b) LDF(12→24): (1,411 + 1,534) / (1,008 + 1,098) = 2,945 / 2,106 = **1.398**  
LDF(24→36): 1,450 / 1,411 = **1.028** (only AY 2022 has both ages)  

(c) AY 2024 ultimate = 1,176 × 1.398 × 1.028 × 1.050 = 1,176 × 1.509 = **$1,775** (approx)  
IBNR = $1,775 − $1,176 = **$599**

**Problem 2:**  
(a) Yes, this is expected. Under claims-made, there is no pure IBNR (unreported claims) because all covered claims must have been reported during the policy year. The very small IBNR reflects only IBNER on known claims — much smaller than the pure IBNR that an occurrence book would carry at the same development age.  
(b) The IBNR represents only IBNER — the expected deficiency in current case reserves on known claims. It should be **much smaller** than a comparable occurrence book because the pure IBNR component is absent. An occurrence book at year-end 2024 would also carry IBNR for unreported events from 2020–2024.  
(c) The large claim settling at 3× case reserve is **IBNER** — reserve development on a known, reported claim. The Bornhuetter-Ferguson method or the development method applied to incurred losses would capture this, as the incurred triangle reflects actual case reserve changes over time.

**Problem 3:**  
(a) Base net IBNR = \$4,200,000 − \$1,600,000 = **\$2,600,000**  
(b) Expected net IBNR = \$2,600,000 + (20% × \$1,600,000) = \$2,600,000 + \$320,000 = **\$2,920,000**  
The 20% probability that the deductible is uncollectible adds \$320,000 to the expected net reserve.  
(c) The actuary should separately identify the **collectibility risk** in the reserve opinion. Per ASOP 43, the actuary should communicate the range of reasonable estimates (from \$2,600,000 assuming full collectibility to \$4,200,000 if the deductible is entirely uncollectible) and the probability-weighted estimate of \$2,920,000. The opinion should note that reserve adequacy depends materially on the insured's financial condition.

</details>